# Batch 2 — F1-Optimized Ensemble + Synthetic Augmentation
### Training: `combined_training.npz` (2200 users) + 300 synthetic anomalies | Test: `second_batch.npz` (860 users)

**Strategy:** All 51 features + 10 synthetic anomaly types + F1-optimized Optuna + Platt calibration

**Models:** XGBoost, CatBoost, LightGBM, Balanced RF → Calibrated Weighted Ensemble

In [ ]:
import pandas as pd
import numpy as np
import zipfile, warnings
import xgboost as xgb
import lightgbm as lgb
import optuna
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import entropy
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score
from sklearn.preprocessing import RobustScaler, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import IsolationForest
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from imblearn.ensemble import BalancedRandomForestClassifier

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

import importlib, feature_pipeline
importlib.reload(feature_pipeline)

from feature_pipeline import (
    combine_labeled_data, load_data, compute_item_stats,
    build_features, get_feature_columns, get_robust_feature_columns,
    generate_synthetic_anomalies
)

print('Imports OK')

## 1. Load & Combine Training Data + Build Features

In [ ]:
# Combine both labeled datasets for maximum training data
combined_path = combine_labeled_data(
    "data/training_batch_with_labels.npz",
    "data/first_batch_with_labels.npz"
)

# Load combined training + test
XX_train, yy, XX_test = load_data(combined_path, test_path="data/second_batch.npz")

print(f"\nTraining interactions: {XX_train.shape[0]:,}")
print(f"Training users: {yy.shape[0]} ({int(yy['label'].sum())} anomalous)")
print(f"Test interactions: {XX_test.shape[0]:,}")
print(f"Test users: {XX_test['user'].nunique()}")

# Augment with synthetic anomalies (10 types x 30 = 300 extra anomalous users)
XX_train, yy = generate_synthetic_anomalies(XX_train, yy, n_per_type=30)

print(f"\nAugmented training interactions: {XX_train.shape[0]:,}")
print(f"Augmented training users: {yy.shape[0]} ({int(yy['label'].sum())} anomalous)")

In [ ]:
# Compute item stats from training ONLY (leakage-safe)
print("Computing item stats + SVD...")
item_stats = compute_item_stats(XX_train, n_svd_components=20)

# Build features for train and test
print("Building training features...")
train_feats = build_features(XX_train, item_stats).merge(yy, on="user")
print("Building test features...")
test_feats = build_features(XX_test, item_stats)

# Use ALL 51 features (adversarial AUC=0.56 confirmed no shift)
feature_cols = get_feature_columns(train_feats)

X_train = train_feats[feature_cols].values
y_train = train_feats["label"].values
X_test = test_feats[feature_cols].values
test_users = test_feats["user"].values

# Track which users are synthetic for visualization
synth_mask = train_feats["user"].values >= (yy["user"].max() - 300 + 1)

# Scale
scaler = RobustScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

print(f"\nFeatures: {len(feature_cols)}")
print(f"Training shape: {X_train_s.shape} ({synth_mask.sum()} synthetic)")
print(f"Test shape: {X_test_s.shape}")
print(f"\nFeature list:")
for i, f in enumerate(feature_cols):
    print(f"  {i+1:2d}. {f}")

## 1b. Adversarial Validation — Detect Train/Test Distribution Shift

In [ ]:
# Adversarial validation: can a classifier distinguish train vs test users?
# High AUC = distribution shift; ~0.5 = distributions match
adv_X = np.vstack([X_train_full, X_test_full])
adv_y = np.array([0]*len(X_train_full) + [1]*len(X_test_full))

adv_model = lgb.LGBMClassifier(n_estimators=100, random_state=42, n_jobs=-1, verbose=-1)
adv_aucs = cross_val_score(adv_model, adv_X, adv_y, cv=5, scoring='roc_auc')
print(f"Adversarial AUC: {adv_aucs.mean():.4f} ± {adv_aucs.std():.4f}")

if adv_aucs.mean() > 0.65:
    print("  → SHIFT DETECTED: some features differ between train/test")
    # Find which features cause the shift
    adv_model.fit(adv_X, adv_y)
    adv_imp = pd.Series(adv_model.feature_importances_, index=feature_cols_full)
    adv_imp = adv_imp.sort_values(ascending=False)
    print("\n  Top 10 train/test discriminating features:")
    for f, v in adv_imp.head(10).items():
        print(f"    {f:30s} importance={v}")
else:
    print("  → No significant shift — problem is anomaly-type difference")

## 1c. Visualize Batch 2: PCA/t-SNE Clusters + Feature Distributions

In [ ]:
# --- PCA on combined train+test, colored by source and label ---
# Impute NaN before PCA
imp = SimpleImputer(strategy='median')
X_all = np.vstack([X_train, X_test])
X_all_imp = imp.fit_transform(X_all)

scaler_viz = StandardScaler()
X_all_scaled = scaler_viz.fit_transform(X_all_imp)

pca = PCA(n_components=2, random_state=42)
pca_coords = pca.fit_transform(X_all_scaled)

n_train = len(X_train)
pca_train = pca_coords[:n_train]
pca_test = pca_coords[n_train:]

# Labels for train users
is_normal = (y_train == 0) & ~synth_mask
is_orig_anom = (y_train == 1) & ~synth_mask
is_synth = synth_mask

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Plot 1: PCA colored by source
ax = axes[0]
ax.scatter(pca_train[is_normal, 0], pca_train[is_normal, 1],
           c='steelblue', alpha=0.3, s=10, label=f'Train normal ({is_normal.sum()})')
ax.scatter(pca_train[is_orig_anom, 0], pca_train[is_orig_anom, 1],
           c='red', alpha=0.6, s=25, label=f'Train anomalous ({is_orig_anom.sum()})')
ax.scatter(pca_train[is_synth, 0], pca_train[is_synth, 1],
           c='orange', alpha=0.4, s=20, label=f'Synthetic ({is_synth.sum()})')
ax.scatter(pca_test[:, 0], pca_test[:, 1],
           c='gray', alpha=0.3, s=10, label=f'Test ({len(pca_test)})')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} var)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} var)')
ax.set_title('PCA: Train vs Test vs Synthetic')
ax.legend(fontsize=9)

# Plot 2: PCA test only, colored by IF anomaly score
iso_test_scores = -IsolationForest(n_estimators=300, contamination=0.09,
                                    random_state=42, n_jobs=-1).fit(
    X_all_scaled[:n_train]).decision_function(X_all_scaled[n_train:])

ax = axes[1]
sc = ax.scatter(pca_test[:, 0], pca_test[:, 1],
                c=iso_test_scores, cmap='RdYlBu_r', alpha=0.6, s=15)
plt.colorbar(sc, ax=ax, label='IF anomaly score')
ax.set_xlabel(f'PC1')
ax.set_ylabel(f'PC2')
ax.set_title('PCA Test Users: colored by IF anomaly score')

plt.tight_layout()
plt.show()

print(f"\nPCA explained variance: {pca.explained_variance_ratio_[:2].sum():.1%} (2 components)")

In [ ]:
# --- t-SNE on combined train+test ---
print("Running t-SNE (may take a minute)...")
tsne = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=1000)
tsne_coords = tsne.fit_transform(X_all_scaled)

tsne_train = tsne_coords[:n_train]
tsne_test = tsne_coords[n_train:]

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Plot 1: t-SNE colored by source
ax = axes[0]
ax.scatter(tsne_train[is_normal, 0], tsne_train[is_normal, 1],
           c='steelblue', alpha=0.3, s=10, label=f'Train normal')
ax.scatter(tsne_train[is_orig_anom, 0], tsne_train[is_orig_anom, 1],
           c='red', alpha=0.6, s=25, label=f'Train anomalous')
ax.scatter(tsne_train[is_synth, 0], tsne_train[is_synth, 1],
           c='orange', alpha=0.4, s=20, label=f'Synthetic')
ax.scatter(tsne_test[:, 0], tsne_test[:, 1],
           c='gray', alpha=0.3, s=10, label=f'Test')
ax.set_title('t-SNE: Train vs Test vs Synthetic')
ax.legend(fontsize=9)

# Plot 2: t-SNE test only, colored by IF score
ax = axes[1]
sc = ax.scatter(tsne_test[:, 0], tsne_test[:, 1],
                c=iso_test_scores, cmap='RdYlBu_r', alpha=0.6, s=15)
plt.colorbar(sc, ax=ax, label='IF anomaly score')
ax.set_title('t-SNE Test Users: colored by IF anomaly score')

plt.tight_layout()
plt.show()

In [ ]:
# --- Feature distributions: Train Normal vs Train Anomalous vs Test ---
top_features = ['prop_rating_0', 'prop_zero', 'rating_entropy', 'rating_range',
                'prop_rating_1', 'rating_kurt', 'rating_min', 'rating_mean',
                'rating_std', 'item_coverage_ratio']

train_normal_feats = train_feats[train_feats["label"] == 0][top_features]
train_anom_feats = train_feats[(train_feats["label"] == 1) & ~synth_mask][top_features]
test_feats_viz = test_feats[top_features]

ncols = 5
nrows = 2
fig, axes = plt.subplots(nrows, ncols, figsize=(20, 8))

for i, feat in enumerate(top_features):
    ax = axes[i // ncols, i % ncols]
    sns.kdeplot(train_normal_feats[feat].dropna(), label='Train Normal',
                fill=True, alpha=0.3, ax=ax, color='steelblue')
    sns.kdeplot(train_anom_feats[feat].dropna(), label='Train Anomalous',
                fill=True, alpha=0.3, ax=ax, color='red')
    sns.kdeplot(test_feats_viz[feat].dropna(), label='Test (all)',
                ax=ax, color='green', linestyle='--', linewidth=2)
    ax.set_title(feat, fontsize=10)
    ax.set_ylabel('')
    if i == 0:
        ax.legend(fontsize=7)
    else:
        ax.get_legend().remove() if ax.get_legend() else None

fig.suptitle('Feature Distributions: Train Normal vs Train Anomalous vs Test', fontsize=13)
plt.tight_layout()
plt.show()

print("Look for features where Test (green) diverges from Train Normal (blue)")
print("Those features might reveal where test anomalies hide")

## 1d. Add Unsupervised Anomaly Scores as Extra Features

In [ ]:
# Add Isolation Forest anomaly scores as extra features
# IF captures "structural unusualness" without labels — transfers better across batches
iso = IsolationForest(n_estimators=500, contamination=0.09, random_state=42, n_jobs=-1)
iso.fit(X_train_s)

iso_train = -iso.decision_function(X_train_s).reshape(-1, 1)
iso_test = -iso.decision_function(X_test_s).reshape(-1, 1)

# Append IF scores to feature matrices
X_train_s = np.column_stack([X_train_s, iso_train])
X_test_s = np.column_stack([X_test_s, iso_test])

feature_cols_with_iso = list(feature_cols) + ["iso_forest_score"]

print(f"Added IF anomaly score → final shape: {X_train_s.shape}")
print(f"IF score range (train): [{iso_train.min():.4f}, {iso_train.max():.4f}]")
print(f"IF score range (test):  [{iso_test.min():.4f}, {iso_test.max():.4f}]")

## 2. Cross-Validation Setup + Helper Functions

In [ ]:
SEED = 42
N_FOLDS = 10
spw = np.sum(y_train == 0) / np.sum(y_train == 1)
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

print(f"Class imbalance ratio: {spw:.2f}")
print(f"Anomalous users: {np.sum(y_train == 1)}, Normal: {np.sum(y_train == 0)}")
print(f"Per fold: ~{np.sum(y_train == 1) // N_FOLDS} anomalous in validation")


def find_best_f1_threshold(scores, y_true):
    """Find threshold that maximizes F1 on given scores."""
    thresholds = np.linspace(0.01, 0.99, 500)
    best_f1, best_t = 0, 0.5
    for t in thresholds:
        preds = (scores >= t).astype(int)
        f1 = f1_score(y_true, preds, zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_t = t
    return best_f1, best_t


def evaluate_oof(name, oof_scores, y_true):
    """Evaluate OOF predictions and find best F1 threshold."""
    auc = roc_auc_score(y_true, oof_scores)
    best_f1, best_t = find_best_f1_threshold(oof_scores, y_true)

    preds = (oof_scores >= best_t).astype(int)
    prec = precision_score(y_true, preds, zero_division=0)
    rec = recall_score(y_true, preds, zero_division=0)

    print(f"\n{'='*50}")
    print(f"{name}")
    print(f"{'='*50}")
    print(f"  OOF AUC:     {auc:.4f}")
    print(f"  Best F1:     {best_f1:.4f} (threshold={best_t:.3f})")
    print(f"  Precision:   {prec:.4f}")
    print(f"  Recall:      {rec:.4f}")
    return auc, best_f1, best_t

## 3. Model 1 — XGBoost with Optuna

In [ ]:
def xgb_objective(trial):
    grow_policy = trial.suggest_categorical('grow_policy', ['depthwise', 'lossguide'])
    params = dict(
        n_estimators      = trial.suggest_int('n_estimators', 100, 1500),
        learning_rate     = trial.suggest_float('learning_rate', 0.005, 0.15, log=True),
        subsample         = trial.suggest_float('subsample', 0.5, 0.9),
        colsample_bytree  = trial.suggest_float('colsample_bytree', 0.3, 0.8),
        colsample_bylevel = trial.suggest_float('colsample_bylevel', 0.4, 0.8),
        min_child_weight  = trial.suggest_int('min_child_weight', 5, 30),
        gamma             = trial.suggest_float('gamma', 0.1, 5.0),
        reg_alpha         = trial.suggest_float('reg_alpha', 0.1, 10.0, log=True),
        reg_lambda        = trial.suggest_float('reg_lambda', 1.0, 20.0, log=True),
        max_delta_step    = trial.suggest_int('max_delta_step', 0, 10),
        scale_pos_weight  = trial.suggest_float('scale_pos_weight', spw * 0.5, spw * 3.0),
        grow_policy       = grow_policy,
    )
    if grow_policy == 'lossguide':
        params['max_leaves'] = trial.suggest_int('max_leaves', 16, 48)
    else:
        params['max_depth'] = trial.suggest_int('max_depth', 3, 6)

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=trial.number)
    f1s = []
    for tr_i, val_i in cv.split(X_train_s, y_train):
        m = xgb.XGBClassifier(
            **params, eval_metric='aucpr', early_stopping_rounds=50,
            random_state=SEED, n_jobs=-1, tree_method='hist'
        )
        m.fit(X_train_s[tr_i], y_train[tr_i],
              eval_set=[(X_train_s[val_i], y_train[val_i])], verbose=False)
        proba = m.predict_proba(X_train_s[val_i])[:, 1]
        best_f1 = max(
            f1_score(y_train[val_i], (proba >= t).astype(int), zero_division=0)
            for t in np.linspace(0.05, 0.95, 100)
        )
        f1s.append(best_f1)
    return np.mean(f1s)

print("Running XGBoost Optuna search (150 trials, F1-optimized)...")
xgb_study = optuna.create_study(direction='maximize')
xgb_study.optimize(xgb_objective, n_trials=150, show_progress_bar=True)

xgb_best = xgb_study.best_params
print(f'\nBest XGBoost CV F1: {xgb_study.best_value:.4f}')
print('Best params:', xgb_best)

In [ ]:
# XGBoost OOF predictions (10-fold)
oof_xgb = np.zeros(len(y_train))
test_preds_xgb = np.zeros(len(X_test_s))
xgb_models = []

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train_s, y_train)):
    m = xgb.XGBClassifier(
        **xgb_best, eval_metric='aucpr', early_stopping_rounds=50,
        random_state=SEED, n_jobs=-1, tree_method='hist'
    )
    m.fit(X_train_s[tr_idx], y_train[tr_idx],
          eval_set=[(X_train_s[val_idx], y_train[val_idx])], verbose=False)
    oof_xgb[val_idx] = m.predict_proba(X_train_s[val_idx])[:, 1]
    test_preds_xgb += m.predict_proba(X_test_s)[:, 1] / N_FOLDS
    xgb_models.append(m)
    print(f"  Fold {fold+1:2d}: AUC={roc_auc_score(y_train[val_idx], oof_xgb[val_idx]):.4f}")

xgb_auc, xgb_f1, xgb_thresh = evaluate_oof("XGBoost", oof_xgb, y_train)

## 4. Model 2 — CatBoost with Optuna

In [ ]:
try:
    from catboost import CatBoostClassifier
    HAS_CATBOOST = True
except ImportError:
    HAS_CATBOOST = False
    print("CatBoost not installed. Run: pip install catboost")

if HAS_CATBOOST:
    def cat_objective(trial):
        params = dict(
            iterations     = trial.suggest_int('iterations', 200, 1500),
            learning_rate  = trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
            depth          = trial.suggest_int('depth', 3, 6),
            l2_leaf_reg    = trial.suggest_float('l2_leaf_reg', 3.0, 30.0),
            bagging_temperature = trial.suggest_float('bagging_temperature', 0, 3.0),
            random_strength     = trial.suggest_float('random_strength', 0.5, 5.0),
            border_count        = trial.suggest_int('border_count', 32, 255),
            auto_class_weights  = 'Balanced',
            eval_metric         = 'AUC',
            random_seed         = SEED,
            verbose             = 0,
        )
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=trial.number)
        f1s = []
        for tr_i, val_i in cv.split(X_train_s, y_train):
            m = CatBoostClassifier(**params)
            m.fit(X_train_s[tr_i], y_train[tr_i],
                  eval_set=(X_train_s[val_i], y_train[val_i]),
                  early_stopping_rounds=50, verbose=0)
            proba = m.predict_proba(X_train_s[val_i])[:, 1]
            best_f1 = max(
                f1_score(y_train[val_i], (proba >= t).astype(int), zero_division=0)
                for t in np.linspace(0.05, 0.95, 100)
            )
            f1s.append(best_f1)
        return np.mean(f1s)

    print("Running CatBoost Optuna search (100 trials, F1-optimized)...")
    cat_study = optuna.create_study(direction='maximize')
    cat_study.optimize(cat_objective, n_trials=100, show_progress_bar=True)
    cat_best = cat_study.best_params
    print(f'\nBest CatBoost CV F1: {cat_study.best_value:.4f}')
    print('Best params:', cat_best)

In [ ]:
# CatBoost OOF predictions (10-fold)
if HAS_CATBOOST:
    oof_cat = np.zeros(len(y_train))
    test_preds_cat = np.zeros(len(X_test_s))

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train_s, y_train)):
        m = CatBoostClassifier(
            **cat_best,
            auto_class_weights='Balanced',
            eval_metric='AUC',
            random_seed=SEED,
            verbose=0,
        )
        m.fit(X_train_s[tr_idx], y_train[tr_idx],
              eval_set=(X_train_s[val_idx], y_train[val_idx]),
              early_stopping_rounds=50, verbose=0)
        oof_cat[val_idx] = m.predict_proba(X_train_s[val_idx])[:, 1]
        test_preds_cat += m.predict_proba(X_test_s)[:, 1] / N_FOLDS
        print(f"  Fold {fold+1:2d}: AUC={roc_auc_score(y_train[val_idx], oof_cat[val_idx]):.4f}")

    cat_auc, cat_f1, cat_thresh = evaluate_oof("CatBoost", oof_cat, y_train)
else:
    oof_cat = oof_xgb.copy()
    test_preds_cat = test_preds_xgb.copy()
    cat_f1 = 0
    print("Skipping CatBoost (not installed)")

## 5. Model 3 — LightGBM with Optuna

In [ ]:
def lgb_objective(trial):
    params = dict(
        n_estimators      = trial.suggest_int('n_estimators', 100, 1500),
        learning_rate     = trial.suggest_float('learning_rate', 0.005, 0.15, log=True),
        num_leaves        = trial.suggest_int('num_leaves', 8, 48),
        subsample         = trial.suggest_float('subsample', 0.5, 0.9),
        colsample_bytree  = trial.suggest_float('colsample_bytree', 0.3, 0.8),
        min_child_samples = trial.suggest_int('min_child_samples', 20, 100),
        reg_alpha         = trial.suggest_float('reg_alpha', 0.1, 10.0, log=True),
        reg_lambda        = trial.suggest_float('reg_lambda', 1.0, 20.0, log=True),
        min_gain_to_split = trial.suggest_float('min_gain_to_split', 0.01, 1.0),
        scale_pos_weight  = trial.suggest_float('scale_pos_weight', spw * 0.5, spw * 3.0),
        random_state      = SEED,
        n_jobs            = -1,
        verbose           = -1,
    )
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=trial.number)
    f1s = []
    for tr_i, val_i in cv.split(X_train_s, y_train):
        m = lgb.LGBMClassifier(**params)
        m.fit(X_train_s[tr_i], y_train[tr_i],
              eval_set=[(X_train_s[val_i], y_train[val_i])],
              callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)])
        proba = m.predict_proba(X_train_s[val_i])[:, 1]
        best_f1 = max(
            f1_score(y_train[val_i], (proba >= t).astype(int), zero_division=0)
            for t in np.linspace(0.05, 0.95, 100)
        )
        f1s.append(best_f1)
    return np.mean(f1s)

print("Running LightGBM Optuna search (150 trials, F1-optimized)...")
lgb_study = optuna.create_study(direction='maximize')
lgb_study.optimize(lgb_objective, n_trials=150, show_progress_bar=True)

lgb_best = lgb_study.best_params
print(f'\nBest LightGBM CV F1: {lgb_study.best_value:.4f}')
print('Best params:', lgb_best)

In [ ]:
# LightGBM OOF predictions (10-fold)
oof_lgb = np.zeros(len(y_train))
test_preds_lgb = np.zeros(len(X_test_s))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train_s, y_train)):
    m = lgb.LGBMClassifier(**lgb_best, random_state=SEED, n_jobs=-1, verbose=-1)
    m.fit(X_train_s[tr_idx], y_train[tr_idx],
          eval_set=[(X_train_s[val_idx], y_train[val_idx])],
          callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)])
    oof_lgb[val_idx] = m.predict_proba(X_train_s[val_idx])[:, 1]
    test_preds_lgb += m.predict_proba(X_test_s)[:, 1] / N_FOLDS
    print(f"  Fold {fold+1:2d}: AUC={roc_auc_score(y_train[val_idx], oof_lgb[val_idx]):.4f}")

lgb_auc, lgb_f1, lgb_thresh = evaluate_oof("LightGBM", oof_lgb, y_train)

## 6. Model 4 — Balanced Random Forest

In [ ]:
# Balanced Random Forest — undersamples majority per bootstrap (10-fold)
oof_brf = np.zeros(len(y_train))
test_preds_brf = np.zeros(len(X_test_s))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train_s, y_train)):
    m = BalancedRandomForestClassifier(
        n_estimators=500, max_depth=10, min_samples_leaf=5,
        random_state=SEED, n_jobs=-1
    )
    m.fit(X_train_s[tr_idx], y_train[tr_idx])
    oof_brf[val_idx] = m.predict_proba(X_train_s[val_idx])[:, 1]
    test_preds_brf += m.predict_proba(X_test_s)[:, 1] / N_FOLDS
    print(f"  Fold {fold+1:2d}: AUC={roc_auc_score(y_train[val_idx], oof_brf[val_idx]):.4f}")

brf_auc, brf_f1, brf_thresh = evaluate_oof("Balanced RF", oof_brf, y_train)

## 7. Model Comparison Summary

In [ ]:
# Summary of all 4 models
results = pd.DataFrame({
    'Model': ['XGBoost', 'CatBoost' if HAS_CATBOOST else 'CatBoost (skipped)',
              'LightGBM', 'Balanced RF'],
    'OOF AUC': [roc_auc_score(y_train, oof_xgb),
                roc_auc_score(y_train, oof_cat),
                roc_auc_score(y_train, oof_lgb),
                roc_auc_score(y_train, oof_brf)],
    'OOF F1': [xgb_f1, cat_f1, lgb_f1, brf_f1],
})
results = results.sort_values('OOF F1', ascending=False).reset_index(drop=True)
print(results.to_string(index=False))

## 8. Platt Calibration + Calibrated Weighted Ensemble

In [ ]:
# Platt calibration: fit LogisticRegression on OOF scores → calibrated probabilities
# This ensures scores are well-calibrated so thresholds transfer to test data

def platt_calibrate(oof_scores, test_scores, y_true):
    """Platt scaling: calibrate OOF scores and apply to test."""
    cal = LogisticRegression(C=1e5, max_iter=1000, random_state=SEED)
    cal.fit(oof_scores.reshape(-1, 1), y_true)
    oof_cal = cal.predict_proba(oof_scores.reshape(-1, 1))[:, 1]
    test_cal = cal.predict_proba(test_scores.reshape(-1, 1))[:, 1]
    return oof_cal, test_cal

# Calibrate each model
oof_xgb_cal, test_xgb_cal = platt_calibrate(oof_xgb, test_preds_xgb, y_train)
oof_cat_cal, test_cat_cal = platt_calibrate(oof_cat, test_preds_cat, y_train)
oof_lgb_cal, test_lgb_cal = platt_calibrate(oof_lgb, test_preds_lgb, y_train)
oof_brf_cal, test_brf_cal = platt_calibrate(oof_brf, test_preds_brf, y_train)

print("Calibrated OOF score stats:")
for name, oof_cal in [("XGBoost", oof_xgb_cal), ("CatBoost", oof_cat_cal),
                       ("LightGBM", oof_lgb_cal), ("Balanced RF", oof_brf_cal)]:
    print(f"  {name:12s}: mean={oof_cal.mean():.4f}, >0.5: {(oof_cal > 0.5).sum()}/{len(oof_cal)}")

# F1-weighted ensemble of calibrated probabilities
model_f1s = np.array([xgb_f1, cat_f1, lgb_f1, brf_f1])
weights = model_f1s / model_f1s.sum()

oof_ensemble = (weights[0] * oof_xgb_cal + weights[1] * oof_cat_cal +
                weights[2] * oof_lgb_cal + weights[3] * oof_brf_cal)
test_ensemble = (weights[0] * test_xgb_cal + weights[1] * test_cat_cal +
                 weights[2] * test_lgb_cal + weights[3] * test_brf_cal)

print(f"\nEnsemble weights: XGB={weights[0]:.3f}, CAT={weights[1]:.3f}, "
      f"LGB={weights[2]:.3f}, BRF={weights[3]:.3f}")

ens_auc, ens_f1, ens_thresh = evaluate_oof("Calibrated Weighted Ensemble", oof_ensemble, y_train)

# Stacking meta-learner on calibrated OOF
oof_stack = np.column_stack([oof_xgb_cal, oof_cat_cal, oof_lgb_cal, oof_brf_cal])
test_stack = np.column_stack([test_xgb_cal, test_cat_cal, test_lgb_cal, test_brf_cal])

oof_meta = np.zeros(len(y_train))
test_meta = np.zeros(len(X_test_s))

for fold, (tr_idx, val_idx) in enumerate(skf.split(oof_stack, y_train)):
    meta = LogisticRegression(C=1.0, max_iter=1000, random_state=SEED)
    meta.fit(oof_stack[tr_idx], y_train[tr_idx])
    oof_meta[val_idx] = meta.predict_proba(oof_stack[val_idx])[:, 1]
    test_meta += meta.predict_proba(test_stack)[:, 1] / N_FOLDS

meta_auc, meta_f1, meta_thresh = evaluate_oof("Stacking (calibrated)", oof_meta, y_train)

In [ ]:
# Pick the best approach based on OOF F1 (not AUC)
best_approaches = {
    'XGBoost (calibrated)':       (test_xgb_cal, xgb_f1),
    'CatBoost (calibrated)':      (test_cat_cal, cat_f1),
    'LightGBM (calibrated)':      (test_lgb_cal, lgb_f1),
    'Balanced RF (calibrated)':   (test_brf_cal, brf_f1),
    'Weighted Ensemble':          (test_ensemble, ens_f1),
    'Stacking (calibrated)':      (test_meta, meta_f1),
}

print("\nAll approaches ranked by OOF F1:")
for name, (_, f1) in sorted(best_approaches.items(), key=lambda x: -x[1][1] if isinstance(x[1], tuple) else -x[1]):
    print(f"  {name:30s}: F1={f1:.4f}" if not isinstance(f1, tuple) else f"  {name:30s}: F1={f1:.4f}")

best_name = max(best_approaches, key=lambda k: best_approaches[k][1])
best_test_preds = best_approaches[best_name][0]
best_f1 = best_approaches[best_name][1]
print(f"\n>>> Best approach: {best_name} (OOF F1 = {best_f1:.4f})")

# Sanity check: how many users flagged?
for name, (preds, _) in best_approaches.items():
    n_flagged = (preds > 0.5).sum()
    pct = n_flagged / len(preds) * 100
    print(f"  {name:30s}: {n_flagged}/{len(preds)} flagged ({pct:.1f}%)")
print(f"\n  Expected anomaly rate: ~9% ({int(len(test_preds_xgb)*0.09)} users)")

## 9. Feature Importance (XGBoost)

In [ ]:
# Average feature importance across XGBoost fold models
avg_imp = np.mean([m.feature_importances_ for m in xgb_models], axis=0)
feat_names = list(feature_cols) + ["iso_forest_score"]
feat_imp = pd.Series(avg_imp, index=feat_names).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 14))
feat_imp.plot.barh(ax=ax)
ax.invert_yaxis()
ax.set_title('XGBoost Average Feature Importance (10-fold, 51 features + IF + synthetic augmentation)')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()

print("\nTop 15 features:")
print(feat_imp.head(15).to_string())

## 10. Generate Submission for second_batch

In [ ]:
# Sort predictions by user ID (Codabench expects this order)
test_order = test_feats.sort_values('user').reset_index(drop=True)
sort_idx = test_feats['user'].argsort().values

# Use best approach's predictions, reordered by user ID
y_score = best_test_preds[sort_idx]

# Normalise to [0, 1]
y_score_norm = (y_score - y_score.min()) / (y_score.max() - y_score.min() + 1e-9)

# Save submission
np.savez('submission.npz', predictions=y_score_norm)
with zipfile.ZipFile('submission.zip', 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.write('submission.npz', arcname='submission.npz')

print(f"Submission method: {best_name}")
print(f"Users predicted:   {len(y_score_norm)}")
print(f"First 5 user IDs:  {test_order['user'].values[:5]}")
print(f"Score range:       [{y_score_norm.min():.4f}, {y_score_norm.max():.4f}]")
print(f"Score mean:        {y_score_norm.mean():.4f}")
print(f"Flagged (>0.5):    {(y_score_norm > 0.5).sum()}/{len(y_score_norm)} "
      f"({(y_score_norm > 0.5).sum()/len(y_score_norm)*100:.1f}%)")
print(f"\nsubmission.zip ready for Codabench")

In [ ]:
# Save alternative submissions (3 per week strategy)
import os
os.makedirs('submission_files', exist_ok=True)

alternatives = {
    'xgboost_cal':    test_xgb_cal,
    'catboost_cal':   test_cat_cal,
    'lightgbm_cal':   test_lgb_cal,
    'brf_cal':        test_brf_cal,
    'weighted_ens':   test_ensemble,
    'stacking_cal':   test_meta,
}

for name, preds in alternatives.items():
    p = preds[sort_idx]
    p_norm = (p - p.min()) / (p.max() - p.min() + 1e-9)
    np.savez(f'submission_files/{name}.npz', predictions=p_norm)
    with zipfile.ZipFile(f'submission_files/{name}.zip', 'w', zipfile.ZIP_DEFLATED) as zf:
        zf.write(f'submission_files/{name}.npz', arcname='submission.npz')

print("Alternative submissions saved in submission_files/:")
for name, preds in alternatives.items():
    p = preds[sort_idx]
    p_norm = (p - p.min()) / (p.max() - p.min() + 1e-9)
    n_flagged = (p_norm > 0.5).sum()
    print(f"  {name}.zip  (flagged: {n_flagged}/{len(p_norm)}, {n_flagged/len(p_norm)*100:.1f}%)")

print(f"\nRecommended 3 submissions:")
print(f"  1. Best single model (calibrated)")
print(f"  2. weighted_ens.zip (calibrated weighted ensemble)")
print(f"  3. stacking_cal.zip (stacking on calibrated OOF)")